# Neural Collaborative Filtering (NCF) for Recommendation

## 1. Imports

In [3]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))

In [4]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
import random
import os
from sklearn.metrics import roc_auc_score

from helpers.data_loaders import load_movielens_data, load_steam_data

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")


Using device: cuda


## 2. Data Loading and Preparation

In [2]:
movies_df, ratings_df = load_movielens_data()
reviews_df_steam, items_df_steam = load_steam_data()

ratings_df = ratings_df[ratings_df['rating'] >= 4.0]
movielens_interactions = pd.DataFrame({
    'user_id': 'movielens_user_' + ratings_df['userId'].astype(str),
    'item_id': 'movielens_item_' + ratings_df['movieId'].astype(str)
})

steam_interactions = pd.DataFrame({
    'user_id': 'steam_user_' + reviews_df_steam['user_id'].astype(str),
    'item_id': 'steam_item_' + reviews_df_steam['app_id'].astype(str)
})

all_interactions = pd.concat([movielens_interactions, steam_interactions]).drop_duplicates()

print(f"Total unique interactions: {len(all_interactions)}")

unique_users = all_interactions['user_id'].unique()
unique_items = all_interactions['item_id'].unique()

user_map = {user: i for i, user in enumerate(unique_users)}
item_map = {item: i for i, item in enumerate(unique_items)}

num_users = len(user_map)
num_items = len(item_map)

print(f"Number of users: {num_users}")
print(f"Number of items: {num_items}")

all_interactions['user_idx'] = all_interactions['user_id'].map(user_map)
all_interactions['item_idx'] = all_interactions['item_id'].map(item_map)


Total unique interactions: 12497401
Number of users: 184419
Number of items: 43660


## 2.1 Data Splitting for Evaluation

In [3]:
def split_data(interactions, test_size=0.2):
    test_indices = np.random.choice(interactions.index, size=int(len(interactions) * test_size), replace=False)
    test = interactions.loc[test_indices]
    train = interactions.drop(test_indices)
    return train, test

train_interactions, test_interactions = split_data(all_interactions)


## 3. NCF Model Definition

In [4]:
class NCF(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim_gmf, embedding_dim_mlp, mlp_layers):
        super().__init__()
        
        self.user_embedding_gmf = nn.Embedding(num_users, embedding_dim_gmf)
        self.item_embedding_gmf = nn.Embedding(num_items, embedding_dim_gmf)
        
        self.user_embedding_mlp = nn.Embedding(num_users, embedding_dim_mlp)
        self.item_embedding_mlp = nn.Embedding(num_items, embedding_dim_mlp)
        
        self.mlp = nn.Sequential()
        input_size = 2 * embedding_dim_mlp
        for i, layer_size in enumerate(mlp_layers):
            self.mlp.add_module(f"linear_{i}", nn.Linear(input_size, layer_size))
            self.mlp.add_module(f"relu_{i}", nn.ReLU())
            input_size = layer_size
            
        self.predict_layer = nn.Linear(embedding_dim_gmf + mlp_layers[-1], 1)
        
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.user_embedding_gmf.weight, std=0.01)
        nn.init.normal_(self.item_embedding_gmf.weight, std=0.01)
        nn.init.normal_(self.user_embedding_mlp.weight, std=0.01)
        nn.init.normal_(self.item_embedding_mlp.weight, std=0.01)
        
        for m in self.mlp.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
        
        nn.init.kaiming_uniform_(self.predict_layer.weight, a=1, nonlinearity='sigmoid')

    def forward(self, user_indices, item_indices):
        user_emb_gmf = self.user_embedding_gmf(user_indices)
        item_emb_gmf = self.item_embedding_gmf(item_indices)
        gmf_output = user_emb_gmf * item_emb_gmf
        
        user_emb_mlp = self.user_embedding_mlp(user_indices)
        item_emb_mlp = self.item_embedding_mlp(item_indices)
        mlp_input = torch.cat([user_emb_mlp, item_emb_mlp], dim=-1)
        mlp_output = self.mlp(mlp_input)
        
        concat = torch.cat([gmf_output, mlp_output], dim=-1)
        
        prediction = self.predict_layer(concat)
        
        return prediction.squeeze()


## 4. Training Setup

In [5]:
class NCFDataset(Dataset):
    def __init__(self, interactions, num_items, num_neg_samples=4, train=True):
        self.interactions = interactions
        self.num_items = num_items
        self.num_neg_samples = num_neg_samples
        self.train = train
        
        self.user_pos_items = self.interactions.groupby('user_idx')['item_idx'].apply(set)
        
        if self.train:
            self.users, self.items, self.labels = self._generate_train_data()

    def _generate_train_data(self):
        users, items, labels = [], [], []
        for _, row in tqdm(self.interactions.iterrows(), desc='Generating training data'):
            user_idx = row['user_idx']
            pos_item_idx = row['item_idx']
            
            users.append(user_idx)
            items.append(pos_item_idx)
            labels.append(1.0)
            
            for _ in range(self.num_neg_samples):
                neg_item_idx = random.randint(0, self.num_items - 1)
                while neg_item_idx in self.user_pos_items.get(user_idx, set()):
                    neg_item_idx = random.randint(0, self.num_items - 1)
                users.append(user_idx)
                items.append(neg_item_idx)
                labels.append(0.0)
        return users, items, labels

    def __len__(self):
        return len(self.users) if self.train else len(self.interactions)

    def __getitem__(self, idx):
        if self.train:
            return self.users[idx], self.items[idx], self.labels[idx]
        else:
            interaction = self.interactions.iloc[idx]
            return interaction['user_idx'], interaction['item_idx']

embedding_dim_gmf = 32
embedding_dim_mlp = 32
mlp_layers = [64, 32, 16]
batch_size = 1024
learning_rate = 1e-3
epochs = 2
num_neg_samples = 4

train_dataset = NCFDataset(train_interactions, num_items, num_neg_samples)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)

model = NCF(num_users, num_items, embedding_dim_gmf, embedding_dim_mlp, mlp_layers).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = nn.BCEWithLogitsLoss()


Generating training data: 0it [00:00, ?it/s]

## 5. Training Loop

In [6]:
model_path = 'models/ncf_model.pth'

if os.path.isfile(model_path):
    model.load_state_dict(torch.load(model_path))
    model.to(device)
    model.eval()
    print(f"Model loaded from {model_path}")
else:
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for user_batch, item_batch, label_batch in progress_bar:
            user_batch = user_batch.to(device)
            item_batch = item_batch.to(device)
            label_batch = label_batch.float().to(device)
            
            optimizer.zero_grad()
            
            predictions = model(user_batch, item_batch)
            
            loss = loss_fn(predictions, label_batch)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            progress_bar.set_postfix({'loss': loss.item()})
            
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs}, Average Loss: {avg_loss:.4f}")
        
    torch.save(model.state_dict(), model_path)
    print(f"Model saved to {model_path}")


Model loaded from models/ncf_model.pth


## 6. Making Recommendations

In [7]:
def get_recommendations(user_id_str, model, top_k=10):
    model.eval()
    
    if user_id_str not in user_map:
        print(f"User '{user_id_str}' not found.")
        return
    user_idx = user_map[user_id_str]
    
    with torch.no_grad():
        user_idx_tensor = torch.LongTensor([user_idx]).to(device)
        item_indices = torch.LongTensor(list(item_map.values())).to(device)
        
        scores = model(user_idx_tensor.repeat(num_items), item_indices)
        
        top_k_scores, top_k_indices = torch.topk(scores, k=top_k)
        
        inv_item_map = {i: item for item, i in item_map.items()}
        
        print(f"Top {top_k} recommendations for user '{user_id_str}':")
        for i, score in zip(top_k_indices.cpu().numpy(), top_k_scores.cpu().numpy()):
            item_id_str = inv_item_map[i]
            if 'steam' in item_id_str:
                item_info = items_df_steam.loc[items_df_steam['app_id'] == int(item_id_str.split('_')[-1]) ]
                if not item_info.empty:
                    print(f"  - [steam] Item: {item_id_str}, Title: {item_info['title'].values[0]} , Score: {score:.4f}")
            else:
                item_info = movies_df.loc[movies_df['movieId'] == int(item_id_str.split('_')[-1]) ]
                if not item_info.empty:
                    print(f"  - [movie] Item: {item_id_str}, Title: {item_info['title'].values[0]} , Score: {score:.4f}")

        print(f"\nLeast preferred {top_k} items for user '{user_id_str}':")
        bottom_k_scores, bottom_k_indices = torch.topk(scores, k=top_k, largest=False)
        for i, score in zip(bottom_k_indices.cpu().numpy(), bottom_k_scores.cpu().numpy()):
            item_id_str = inv_item_map[i]
            if 'steam' in item_id_str:
                item_info = items_df_steam.loc[items_df_steam['app_id'] == int(item_id_str.split('_')[-1]) ]
                if not item_info.empty:
                    print(f"  - [steam] Item: {item_id_str}, Title: {item_info['title'].values[0]} , Score: {score:.4f}")
            else:
                item_info = movies_df.loc[movies_df['movieId'] == int(item_id_str.split('_')[-1]) ]
                if not item_info.empty:
                    print(f"  - [movie] Item: {item_id_str}, Title: {item_info['title'].values[0]} , Score: {score:.4f}")

sample_user_id = 'movielens_user_2'
print(f"Items liked by the user ({sample_user_id}):")
liked_items = all_interactions[all_interactions['user_id'] == sample_user_id]['item_id']
for item in liked_items:
    if 'movielens' in item:
        item_info = movies_df.loc[movies_df['movieId'] == int(item.split('_')[-1]) ]
        if not item_info.empty:
            print(f"  - {item}, Title: {item_info['title'].values[0]}")

get_recommendations(sample_user_id, model)


Items liked by the user (movielens_user_2):
  - movielens_item_110, Title: Braveheart (1995)
  - movielens_item_150, Title: Apollo 13 (1995)
  - movielens_item_151, Title: Rob Roy (1995)
  - movielens_item_236, Title: French Kiss (1995)
  - movielens_item_260, Title: Star Wars: Episode IV - A New Hope (1977)
  - movielens_item_318, Title: Shawshank Redemption, The (1994)
  - movielens_item_333, Title: Tommy Boy (1995)
  - movielens_item_349, Title: Clear and Present Danger (1994)
  - movielens_item_356, Title: Forrest Gump (1994)
  - movielens_item_364, Title: Lion King, The (1994)
  - movielens_item_457, Title: Fugitive, The (1993)
  - movielens_item_497, Title: Much Ado About Nothing (1993)
  - movielens_item_527, Title: Schindler's List (1993)
  - movielens_item_534, Title: Shadowlands (1993)
  - movielens_item_589, Title: Terminator 2: Judgment Day (1991)
  - movielens_item_733, Title: Rock, The (1996)
  - movielens_item_914, Title: My Fair Lady (1964)
  - movielens_item_953, Title

## 7. Evaluation

In [8]:
def evaluate(model, test_interactions, train_interactions, k=20):
    model.eval()
    
    test_user_indices = test_interactions['user_idx'].unique()
    
    test_ground_truth = test_interactions.groupby('user_idx')['item_idx'].apply(list).to_dict()
    train_ground_truth = train_interactions.groupby('user_idx')['item_idx'].apply(list).to_dict()

    recalls, precisions, f1_scores = [], [], []

    with torch.no_grad():
        for user_idx in tqdm(test_user_indices, desc='Evaluating'):
            ground_truth_items = test_ground_truth.get(user_idx, [])
            if not ground_truth_items:
                continue

            excluded_items = train_ground_truth.get(user_idx, [])
            
            user_idx_tensor = torch.LongTensor([user_idx]).to(device)
            item_indices = torch.LongTensor(list(range(num_items))).to(device)

            scores = model(user_idx_tensor.repeat(num_items), item_indices)
            scores[excluded_items] = -np.inf

            _, top_k_indices = torch.topk(scores, k=k)
            top_k_indices = top_k_indices.cpu().numpy()

            hits = np.isin(top_k_indices, ground_truth_items)
            num_hits = np.sum(hits)

            recall = num_hits / len(ground_truth_items)
            precision = num_hits / k
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

            recalls.append(recall)
            precisions.append(precision)
            f1_scores.append(f1)

    avg_recall = np.mean(recalls)
    avg_precision = np.mean(precisions)
    avg_f1 = np.mean(f1_scores)

    print(f'Recall@{k}: {avg_recall:.4f}')
    print(f'Precision@{k}: {avg_precision:.4f}')
    print(f'F1-score@{k}: {avg_f1:.4f}')

    return avg_recall, avg_precision, avg_f1

evaluate(model, test_interactions, train_interactions, k=20)


Evaluating:   0%|          | 0/168310 [00:00<?, ?it/s]

Recall@20: 0.2704
Precision@20: 0.1341
F1-score@20: 0.1417


(np.float64(0.27037704903230175),
 np.float64(0.13409541916701323),
 np.float64(0.1416871663410238))

## 7.1 Link Prediction Evaluation (AUC)

In [9]:
def evaluate_auc(model, test_loader, num_neg_samples=100):
    model.eval()
    auc_scores = []
    
    user_pos_items_all = all_interactions.groupby('user_idx')['item_idx'].apply(set)

    with torch.no_grad():
        for user_idx, pos_item_idx in tqdm(test_loader, desc='Evaluating AUC'):
            
            pos_items = pos_item_idx.tolist()
            user_items = user_pos_items_all.get(user_idx.item(), set())

            neg_items = []
            while len(neg_items) < num_neg_samples:
                neg_item = random.randint(0, num_items - 1)
                if neg_item not in user_items:
                    neg_items.append(neg_item)

            item_indices = torch.LongTensor(pos_items + neg_items).to(device)
            user_indices = torch.LongTensor([user_idx.item()] * len(item_indices)).to(device)
            
            predictions = model(user_indices, item_indices).cpu().numpy()
            labels = np.array([1]*len(pos_items) + [0]*len(neg_items))

            if len(np.unique(labels)) > 1:
                auc = roc_auc_score(labels, predictions)
                auc_scores.append(auc)

    avg_auc = np.mean(auc_scores) if auc_scores else 0
    print(f'Average AUC: {avg_auc:.4f}')
    return avg_auc

test_dataset_auc = NCFDataset(test_interactions, num_items, train=False)
test_loader_auc = DataLoader(test_dataset_auc, batch_size=1, shuffle=True)

evaluate_auc(model, test_loader_auc)


Evaluating AUC:   0%|          | 0/2509004 [00:00<?, ?it/s]

Average AUC: 0.9892


np.float64(0.9892413045176491)